# SQL Subqueries — Runnable Examples

SQLite examples using fictional data. Run cells in order.

In [1]:
import sqlite3
import pandas as pd
connection = sqlite3.connect(":memory:")
print("SQLite ready")

SQLite ready


## 1. Create sample tables

In [2]:
connection.executescript('''
CREATE TABLE departments (department_id INTEGER PRIMARY KEY, department_name TEXT, location TEXT);
INSERT INTO departments VALUES (1,'Engineering','Indore'),(2,'Sales','Pune'),(3,'HR','Mumbai'),(4,'Finance','Indore'),(5,'Research','Pune');
CREATE TABLE employees (employee_id INTEGER PRIMARY KEY, employee_name TEXT, department_id INTEGER, salary REAL, location TEXT);
INSERT INTO employees VALUES
(1,'Asha',1,72000,'Indore'),(2,'Ravi',1,68000,'Pune'),(3,'Meera',1,61000,'Indore'),
(4,'Kabir',2,85000,'Pune'),(5,'Neha',2,52000,'Mumbai'),(6,'Arjun',2,48000,'Indore'),
(7,'Isha',3,78000,'Mumbai'),(8,'Dev',3,42000,'Pune'),(9,'Tara',4,65000,'Indore'),
(10,'Omar',4,58000,'Mumbai'),(11,'Zoya',4,63000,'Pune'),(12,'Nikhil',4,NULL,'Indore');
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_name TEXT, customer_id INTEGER, employee_id INTEGER, amount REAL, status TEXT);
INSERT INTO orders VALUES
(101,'Customer A',1,1,250,'completed'),(102,'Customer B',2,1,125,'completed'),
(103,'Customer C',3,4,600,'cancelled'),(104,'Customer D',4,3,80,'pending'),
(105,'Customer E',5,5,320,'completed'),(106,'Customer F',6,2,180,'completed');
CREATE TABLE products (product_id INTEGER PRIMARY KEY, product_name TEXT, category TEXT);
INSERT INTO products VALUES (1,'Laptop','Electronics'),(2,'Notebook','Stationery'),(3,'Bottle','Accessories');
''')
print("Sample tables created")

Sample tables created


## 2. Scalar subquery: above overall average

In [3]:
pd.read_sql_query('SELECT employee_name, salary FROM employees WHERE salary > (SELECT AVG(salary) FROM employees) ORDER BY salary DESC', connection)

,employee_name,salary
0,Kabir,85000.0
1,Isha,78000.0
2,Asha,72000.0
3,Ravi,68000.0
4,Tara,65000.0
5,Zoya,63000.0


## 3. IN subquery

In [4]:
pd.read_sql_query("SELECT employee_name FROM employees WHERE department_id IN (SELECT department_id FROM departments WHERE location = 'Indore')", connection)

,employee_name
0,Asha
1,Ravi
2,Meera
3,Tara
4,Omar
5,Zoya
6,Nikhil


## 4. EXISTS

In [5]:
pd.read_sql_query('SELECT d.department_name FROM departments d WHERE EXISTS (SELECT 1 FROM employees e WHERE e.department_id = d.department_id)', connection)

,department_name
0,Engineering
1,Sales
2,HR
3,Finance


## 5. NOT EXISTS

In [6]:
pd.read_sql_query('SELECT d.department_name FROM departments d WHERE NOT EXISTS (SELECT 1 FROM employees e WHERE e.department_id = d.department_id)', connection)

,department_name
0,Research


## 6. Correlated subquery: above department average

In [7]:
pd.read_sql_query('SELECT e.employee_name, e.salary FROM employees e WHERE e.salary > (SELECT AVG(e2.salary) FROM employees e2 WHERE e2.department_id = e.department_id) ORDER BY e.salary DESC', connection)

,employee_name,salary
0,Kabir,85000.0
1,Isha,78000.0
2,Asha,72000.0
3,Ravi,68000.0
4,Tara,65000.0
5,Zoya,63000.0


## 7. Subquery in SELECT

In [8]:
pd.read_sql_query('SELECT d.department_name, (SELECT COUNT(*) FROM employees e WHERE e.department_id = d.department_id) AS employee_count FROM departments d', connection)

,department_name,employee_count
0,Engineering,3
1,Sales,3
2,HR,2
3,Finance,4
4,Research,0


## 8. Derived table in FROM

In [9]:
pd.read_sql_query('SELECT department_id, average_salary FROM (SELECT department_id, AVG(salary) AS average_salary FROM employees GROUP BY department_id) AS department_stats WHERE average_salary > 60000', connection)

,department_id,average_salary
0,1,67000.000000
1,2,61666.666667
2,4,62000.000000


## 9. Orders above average amount

In [10]:
pd.read_sql_query('SELECT order_id, customer_name, amount FROM orders WHERE amount > (SELECT AVG(amount) FROM orders) ORDER BY amount DESC', connection)

,order_id,customer_name,amount
0,103,Customer C,600.0
1,105,Customer E,320.0


## 10. Payroll above average department payroll

In [11]:
pd.read_sql_query('SELECT department_id, total_payroll FROM (SELECT department_id, SUM(salary) AS total_payroll FROM employees GROUP BY department_id) p WHERE total_payroll > (SELECT AVG(total_payroll) FROM (SELECT SUM(salary) AS total_payroll FROM employees GROUP BY department_id) x)', connection)

,department_id,total_payroll
0,1,201000.0
1,2,185000.0
2,4,186000.0


## 11. Employees sharing a salary

In [12]:
pd.read_sql_query('SELECT e.employee_name, e.salary FROM employees e WHERE EXISTS (SELECT 1 FROM employees e2 WHERE e2.salary = e.salary AND e2.employee_id <> e.employee_id)', connection)

,employee_name,salary


## Key takeaways

- Scalar subqueries return one value; multi-row subqueries can return many rows.
- `IN` checks membership; `EXISTS` checks for matching rows.
- Correlated subqueries reference outer-query columns.
- `NOT EXISTS` avoids the common `NOT IN` and NULL trap.
- Compare query plans before drawing performance conclusions.